In [1]:
import itertools
import math
from collections import Counter, defaultdict

import rdkit
from IPython.core.interactiveshell import InteractiveShell
from rdkit import Chem

from stereomolgraph import MolGraph, StereoMolGraph
from stereomolgraph.algorithms.color_refine import color_refine_mg, color_refine_smg

InteractiveShell.ast_node_interactivity = "all"

# Optional in newer environments where line_profiler may not be installed
try:
    get_ipython().run_line_magic("load_ext", "line_profiler")
except Exception:
    pass

In [2]:
fullerene_smiles = "C12=C3C4=C5C6=C1C7=C8C9=C1C%10=C%11C(=C29)C3=C2C3=C4C4=C5C5=C9C6=C7C6=C7C8=C1C1=C8C%10=C%10C%11=C2C2=C3C3=C4C4=C5C5=C%11C%12=C(C6=C95)C7=C1C1=C%12C5=C%11C4=C3C3=C5C(=C81)C%10=C23"
taxol_smiles = "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C"
benzene_smiles = "c1ccccc1"
cyclopropane = "C1CC1"
rdmol = Chem.MolFromSmiles(taxol_smiles)
rdmol = Chem.AddHs(rdmol)
# mg = MolGraph.from_rdmol(rdmol)
smg = StereoMolGraph.from_rdmol(rdmol)

smg

# _ = smg.relabel_atoms({a:random.randint(0, 999) for a in smg.atoms}, copy=False)

Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
StereoMolGraph
Atoms
(0 C) (1 C) (2 C) (3 C) (4 C) (5 O) (6 C) (7 C) (8 C) (9 C) (10 C)
(11 C) (12 C) (13 C) (14 C) (15 C) (16 C) (17 C) (18 C) (19 O) (20 C) (21 O)
(22 C) (23 C) (24 C) (25 C) (26 C) (27 C) (28 C) (29 C) (30 N) (31 C) (32 O)
(33 C) (34 C) (35 C) (36 C) (37 C) (38 C) (39 O) (40 O) (41 O) (42 C) (43 O)
(44 C) (45 C) (46 C) (47 C) (48 C) (49 C) (50 C) (51 O) (52 O) (53 C) (54 O)
(55 C) (56 O) (57 C) (58 O) (59 C) (60 O) (61 C) (62 H) (63 H) (64 H) (65 H)
(66 H) (67 H) (68 H) (69 H) (70 H) (71 H) (72 H) (73 H) (74 H) (75 H) (76 H)
(77 H) (78 H) (79 H) (80 H) (81 H) (82 H) (83 H) (84 H) (85 H) (86 H) (8

In [ ]:
from collections.abc import Mapping
from typing import NamedTuple

import numpy as np
from traitlets.traitlets import Any

from stereomolgraph import AtomId
from stereomolgraph.graphs.scrg import Change, Stereo

# TODO:
# define priority based labels:
# - rearest color
# - finishes stereodescriptor
# - degree


class _Parameters(NamedTuple):
    g1_nbrhd: Mapping[AtomId, set[AtomId]]

    g1_labels: Mapping[AtomId, np.int64]

    nodes_of_g1Labels: Mapping[np.int64, set[AtomId]]

    g1_degree: Mapping[AtomId, int]

    # g1_stereo: Mapping[AtomId, list[Stereo]]
    # g1_stereo_changes: Mapping[AtomId, Mapping[Change, list[Stereo]]]


class _CanonState(NamedTuple):
    best: list[Any]
    current: list[Any]

    to_backtrack: list[list[AtomId]]

    included: set[AtomId]

    frontier: set[AtomId]

    external: set[AtomId]

In [ ]:
def initialize(g: MolGraph) -> tuple[_Parameters, _CanonState]:
    """Prepare immutable inputs and initial state for non-stereo canonical enumeration."""
    g1_nbrhd = g.neighbors

    # Empty graph: return a trivially initialized state.
    if len(g1_nbrhd) == 0:
        params = _Parameters(
            g1_nbrhd={},
            g1_labels={},
            nodes_of_g1Labels={},
            g1_degree={},
        )
        state = _CanonState(
            best=[],
            current=[],
            to_backtrack=[],
            included=set(),
            frontier=set(),
            external=set(),
        )
        return params, state

    # MolGraph refinement does not include stereochemistry.
    label_array = color_refine_mg(g)
    g1_labels = {atom: np.int64(label) for atom, label in zip(g.atoms, label_array)}

    nodes_of_g1Labels: defaultdict[np.int64, set[AtomId]] = defaultdict(set)
    for atom, label in g1_labels.items():
        nodes_of_g1Labels[label].add(atom)

    g1_degree = {atom: len(nbrs) for atom, nbrs in g1_nbrhd.items()}

    # MolGraph has no stereo information; keep shape-compatible empty maps.
    g1_stereo: defaultdict[AtomId, list[Stereo]] = defaultdict(list)
    g1_stereo_changes: defaultdict[AtomId, defaultdict[Change, list[Stereo]]] = (
        defaultdict(lambda: defaultdict(list))
    )

    for atom in g1_nbrhd:
        _ = g1_stereo[atom]
        _ = g1_stereo_changes[atom]

    params = _Parameters(
        g1_nbrhd=g1_nbrhd,
        g1_labels=g1_labels,
        nodes_of_g1Labels=dict(nodes_of_g1Labels),
        g1_degree=g1_degree,
    )
    state = _CanonState(
        best=[],
        current=[],
        to_backtrack=[],
        included=set(),
        frontier=set(),
        external=set(g1_nbrhd),
    )
    return params, state

In [3]:
# colors are skipped during enumeration
def log_state(name_str: str, env: dict):
    return
    print(
        name_str,
        f"path {env['current_path']}",
        f"best_path {env['best_path']}",
        f"frontier {env['frontier']}",
        f"label {env['label']}",
        f"current_path_label {env['current_path_labels']}",
        f"best_path_label {env['best_path_labels']}",
        f"missing_branches {env['missing_branches']}",
        f"candidates {env['candidates']}",
        f"counter {env['counter']}",
        "",
        sep="\n",
    )


def count_forward_with_skip(i: int, skip_set: set[int] = frozenset()) -> int:
    i += 1
    while i in skip_set:
        i += 1
    return i


def count_backward_with_skip(i: int, skip_set: set[int] = frozenset()) -> int:
    i -= 1
    while i in skip_set:
        i -= 1
    return i


def func(smg: StereoMolGraph, color_refine: bool = False) -> dict[int, int]:
    "Best-first-search with backtracking for canonical atom enumeration"
    n_atoms = smg.n_atoms  # int
    neighbors = smg._neighbors  # dict[int, set[int]]
    # stereo = smg._bond_stereo | smg._atom_stereo

    # atom_id, color : dict[int, int]
    # color_refine_mg returns a numpy array ordered by smg.atoms
    color_array = color_refine_mg(smg)
    colors = {atom: int(color) for atom, color in zip(smg.atoms, color_array)}
    numbers = defaultdict(lambda: math.inf)  # default to color is no number is assigned

    # skip_set = set(colors.values()) # : set[int]
    # count_forward = functools.partial(count_forward_with_skip,
    #                                  skip_set=skip_set)
    # count_backward = functools.partial(count_backward_with_skip,
    #                                   skip_set=skip_set)

    candidates = {k for k, v in colors.items() if v == max(colors.values())}
    # all atoms which could be explored next, independent of color
    frontier: set[int] = set()
    label = -math.inf

    best_path = []
    best_path_labels = []

    current_path_labels = []
    current_path = []

    missing_branches: list[set[int]] = []
    counter = 0
    # log_state("initialize",locals())
    # best_first_search
    while candidates:
        ### put next candidate into path ###
        atom_id = candidates.pop()

        frontier.discard(atom_id)
        missing_branches.append(candidates)

        current_path.append(atom_id)
        current_path_labels.append(label)

        numbers[current_path[-1]] = counter

        counter += 1

        ### add neighbors to frontier ###
        frontier.update(neighbors[atom_id].difference(current_path))

        # log_state("update_path",locals()) # logging !

        ### update best path ###
        if len(current_path_labels) > len(best_path_labels) or (
            len(current_path_labels) == len(best_path_labels)
            and current_path_labels[-1] < best_path_labels[-1]
        ):
            best_path = current_path.copy()
            best_path_labels = current_path_labels.copy()

        ### forward tracking: frontier to new candidates ###
        if frontier:
            # TODO: include Stereo!
            # TODO: make comparison more efficient!

            hashed_items = []
            for c in frontier:
                neighbor_numbers = [(numbers[nbr], colors[nbr]) for nbr in neighbors[c]]
                neighbor_numbers.sort()

                stereos = []
                for s in smg.stereo:
                    ...

                item_hash = (colors[c], neighbor_numbers, stereos)
                hashed_items.append((item_hash, c))

            hashed_items.sort(key=lambda x: x[0])

            # Get the first group of items with the same hash
            label, candidate_groups = next(
                itertools.groupby(hashed_items, key=lambda x: x[0])
            )
            # print(label)
            candidates = {atom_id for _, atom_id in candidate_groups}

        ### backtacking: missing_branches to candidates ###
        elif not frontier and len(current_path) == n_atoms:
            while not candidates and missing_branches:
                candidates = missing_branches.pop()

                rm_atom_id = current_path.pop()

                if len(current_path) != 0:
                    frontier.add(rm_atom_id)

                label = current_path_labels.pop()

                for nrb in neighbors[rm_atom_id]:
                    if neighbors[nrb].isdisjoint(
                        current_path
                    ):  # or nrb in current_path
                        frontier.discard(nrb)  # TODO: put remove?

                numbers.pop(rm_atom_id)
                counter -= 1

            ### check convergence ###
            if not candidates and not missing_branches:
                return {atom: i for i, atom in enumerate(best_path)}

        # elif not frontier and len(current_path) != n_atoms:
        #    ... # TODO: handle non connected components
        else:
            raise Exception("This should not happen")

    raise Exception("This should not happen")


In [ ]:
# smg = StereoMolGraph.from_rdmol(rdmol)
# %timeit func(smg)
# %lprun -f func func(smg)

In [ ]:
import random

import rdkit.Chem

rdmol: rdkit.Chem.Mol

for i in range(3):
    aids = list(range(rdmol.GetNumAtoms()))
    random.shuffle(aids)
    rdmol = Chem.RenumberAtoms(rdmol, aids)
    smg = StereoMolGraph.from_rdmol(rdmol)

    # smg

    # Counter(color_refine_smg(smg, max_iter=1))
    # Counter(color_refine_smg(smg, max_iter=2))
    # Counter(color_refine_smg(smg, max_iter=3))

    # _ = smg.relabel_atoms({a:random.randint(0, 15) for a in smg.atoms},
    #                     copy=False)
    # label_hash(smg)
    canon_num = func(smg)
    print(canon_num)

    # _ = smg.relabel_atoms(canon_num, copy=False)
    smg
    Counter(map(int, color_refine_smg(smg)))
    # _ = Counter(color_refine_smg(smg, max_iter=0))
    #
# %timeit func()
# %prun [func() for i in range(100)]
# %lprun -f func func()

Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo
Aromatic Bond Stereo


KeyboardInterrupt: 

In [ ]:
import numpy as np

In [ ]:
np.exp(3)